# 2 — Weekly Forecasting EDA, Feature Engineering Experiments & Split

## Objective

This notebook replaces the old severity-classification feature experiments with forecasting-oriented time-series analysis. It consumes Notebook 1's sparse `week_start × state` counts and produces a complete weekly panel and national series.

**Weekly grouping rule:** `week_id` is a sequential integer group identifier beginning at 1. Every date in the same Monday-start local-calendar week belongs to the same group. `week_start` is retained only as a date label and calendar-feature source; the integer `week_id` is the ordered batching key.

All reports with a valid `Start_Time` are retained, including reports whose state is unknown (the `__UNKNOWN__` group). No severity class is predicted.

The chronological 80:20 train/test split is performed at the very end. Notebooks 3I and 3N use the same split metadata but build different representations. Any learned scaling/imputation is fitted later using training data only.

## 0. Dependencies and shared paths

In [ ]:
from pathlib import Path
import os, json, warnings, gc, time, copy, random, sys
import numpy as np
import pandas as pd

SEED = 34
random.seed(SEED)
np.random.seed(SEED)

ON_KAGGLE = Path('/kaggle/working').exists()
PROJECT_ROOT = Path('/kaggle/working/us_accidents_weekly_forecasting') if ON_KAGGLE else Path('./us_accidents_weekly_forecasting')
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
MODEL_DIR = PROJECT_ROOT / 'models'
REPORT_DIR = PROJECT_ROOT / 'reports'
for _p in (PROJECT_ROOT, ARTIFACT_DIR, MODEL_DIR, REPORT_DIR):
    _p.mkdir(parents=True, exist_ok=True)

# Set this only when more than one prior notebook output is attached as a Kaggle input.
ARTIFACT_INPUT_ROOT = None  # e.g. Path('/kaggle/input/us-accidents-weekly-stage-2')

def resolve_artifact(filename, required=True):
    candidates = []
    if ARTIFACT_INPUT_ROOT is not None:
        p = Path(ARTIFACT_INPUT_ROOT) / filename
        if p.exists():
            candidates.append(p)
    p = ARTIFACT_DIR / filename
    if p.exists():
        candidates.append(p)
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.extend(kaggle_input.rglob(filename))
    # Deduplicate resolved paths and prefer the most recently modified attached artifact.
    unique = {str(p.resolve()): p for p in candidates}
    candidates = list(unique.values())
    if not candidates:
        if required:
            raise FileNotFoundError(
                f'Could not locate {filename}. Add the preceding notebook output as a Kaggle input, '
                f'or set ARTIFACT_INPUT_ROOT. Local output searched: {ARTIFACT_DIR}'
            )
        return None
    chosen = max(candidates, key=lambda p: p.stat().st_mtime)
    if len(candidates) > 1:
        print(f'Found {len(candidates)} copies of {filename}; using newest: {chosen}')
    return chosen

print('Project root:', PROJECT_ROOT.resolve())
print('Seed:', SEED)

import matplotlib.pyplot as plt

SPARSE_PATH = resolve_artifact('weekly_state_counts_sparse.parquet')
SOURCE_META_PATH = resolve_artifact('source_aggregation_metadata.json')
source_meta = json.loads(SOURCE_META_PATH.read_text())
weekly_sparse = pd.read_parquet(SPARSE_PATH)
weekly_sparse['week_start'] = pd.to_datetime(weekly_sparse['week_start']).dt.tz_localize(None)
weekly_sparse['state'] = weekly_sparse['state'].fillna('__UNKNOWN__').astype(str)
weekly_sparse['accident_count'] = weekly_sparse['accident_count'].astype('int64')
print(weekly_sparse.head())
print('Sparse aggregated reports:', int(weekly_sparse.accident_count.sum()))

## 1. Build the complete weekly index and sequential `week_id`

We retain the first and last observed calendar weeks. If either is a partial exposure period, `week_days_within_source_bounds` marks it; no accident records are removed. All internal weeks have seven calendar days of exposure.

In [ ]:
first_week = weekly_sparse['week_start'].min().normalize()
last_week = weekly_sparse['week_start'].max().normalize()
week_starts = pd.date_range(first_week, last_week, freq='W-MON')
first_valid_ts = pd.to_datetime(source_meta.get('first_valid_timestamp'))
last_valid_ts = pd.to_datetime(source_meta.get('last_valid_timestamp'))
if pd.isna(first_valid_ts) or pd.isna(last_valid_ts):
    raise ValueError('Notebook 1 metadata is missing valid timestamp bounds; rerun Notebook 1.')

def calendar_days_within_source_bounds(week_start):
    week_start = pd.Timestamp(week_start).normalize()
    observed_start = max(week_start, first_valid_ts.normalize())
    observed_end = min(week_start + pd.Timedelta(days=6), last_valid_ts.normalize())
    return max(0, int((observed_end - observed_start).days) + 1)

if len(week_starts) < 80:
    raise ValueError(f'Only {len(week_starts)} weekly groups found; expected a multi-year time series.')

states = sorted(weekly_sparse['state'].unique().tolist())
week_index = pd.DataFrame({
    'week_start': week_starts,
    'week_id': np.arange(1, len(week_starts) + 1, dtype=np.int32),
})
week_index['week_days_within_source_bounds'] = [calendar_days_within_source_bounds(d) for d in week_index['week_start']]
if (week_index['week_days_within_source_bounds'] <= 0).any():
    raise AssertionError('A generated week falls outside the valid source timestamp interval.')
week_map = dict(zip(week_index['week_start'], week_index['week_id']))

# This field only identifies calendar days lying inside the global first/last timestamp
# bounds. It can flag truncated first/last weeks, but it does NOT certify collection completeness
# inside a week; the source dataset documents possible collection gaps. No accident row is removed.
weekly_sparse['week_start'] = weekly_sparse['week_start'].dt.normalize()
weekly_sparse['week_id'] = weekly_sparse['week_start'].map(week_map).astype('int32')
if weekly_sparse['week_id'].isna().any():
    raise AssertionError('A sparse group did not map to the constructed weekly index.')

full_index = pd.MultiIndex.from_product(
    [week_index['week_id'].tolist(), states], names=['week_id', 'state']
)
panel = (
    weekly_sparse.groupby(['week_id', 'state'], as_index=False)['accident_count'].sum()
    .set_index(['week_id', 'state'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)
panel['week_start'] = panel['week_id'].map(dict(zip(week_index.week_id, week_index.week_start)))
panel['accident_count'] = panel['accident_count'].astype('int64')
panel['week_days_within_source_bounds'] = panel['week_id'].map(dict(zip(week_index.week_id, week_index.week_days_within_source_bounds))).astype('int8')
panel = panel[['week_id', 'week_start', 'state', 'accident_count', 'week_days_within_source_bounds']]
panel = panel.sort_values(['week_id', 'state']).reset_index(drop=True)

national = (
    panel.groupby(['week_id', 'week_start', 'week_days_within_source_bounds'], as_index=False)['accident_count']
    .sum().rename(columns={'accident_count': 'national_accident_count'})
    .sort_values('week_id').reset_index(drop=True)
)
if int(national['national_accident_count'].sum()) != int(weekly_sparse['accident_count'].sum()):
    raise AssertionError('National totals do not conserve all valid timestamped source reports.')

print('Weeks:', len(week_index), '| states:', len(states), '| panel rows:', len(panel))
print('All reports conserved:', f"{int(national.national_accident_count.sum()):,}")
print('Week ID range:', int(week_index.week_id.min()), 'to', int(week_index.week_id.max()))

## 2. Exploratory analysis of the weekly target

Inspect the long-term trend, annual seasonality, distribution and year-over-year pattern. These plots guide feature hypotheses; they are not model-selection results.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(national['week_start'], national['national_accident_count'], linewidth=1.2)
ax.set_title('Weekly reported US accident counts')
ax.set_xlabel('Week starting (Monday)')
ax.set_ylabel('Accident reports')
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(national['national_accident_count'], bins=35)
ax.set_title('Distribution of weekly national accident counts')
ax.set_xlabel('Accident reports per week'); ax.set_ylabel('Number of weeks')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

nat = national.copy()
nat['iso_week'] = nat['week_start'].dt.isocalendar().week.astype(int)
nat['year'] = nat['week_start'].dt.year
seasonal = nat.groupby('iso_week')['national_accident_count'].agg(['mean','median','std']).reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(seasonal.iso_week, seasonal['mean'], label='Mean')
ax.plot(seasonal.iso_week, seasonal['median'], label='Median')
ax.set_title('Weekly seasonality across years')
ax.set_xlabel('ISO week of year'); ax.set_ylabel('Accident reports')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(national['national_accident_count'].describe())

## 3. Candidate forecasting feature map

The following candidates are available at the forecast origin (end of week *t*) when predicting week *t+1*:

- Historical national counts: lags 1, 2, 3, 4, 8, 13, 26 and 52 weeks.
- Rolling mean and standard deviation over 4, 13, 26 and 52 preceding observations, shifted so the target week is never included.
- One-week relative change, computed from already observed counts.
- Known target-week calendar variables: ISO week-of-year sine/cosine and a sequential time index.

The Stream I feature table is constructed in Notebook 3I. The Stream N inputs are state-level weekly count sequences plus calendar and state identity covariates in Notebook 3N. No event-level weather or severity fields are used as future predictors.

In [ ]:
# Visualize the raw national count series with lag references for inspection.
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(national.week_id, national.national_accident_count, label='Observed weekly count', linewidth=1.2)
ax.plot(national.week_id, national.national_accident_count.shift(1), label='Lag 1', alpha=0.7)
ax.plot(national.week_id, national.national_accident_count.shift(52), label='Lag 52', alpha=0.7)
ax.set_title('Observed weekly counts and candidate historical references')
ax.set_xlabel('Sequential week_id'); ax.set_ylabel('Accident reports')
ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

# State contribution: diagnostic only. The national target remains the sum of all states.
state_totals = panel.groupby('state')['accident_count'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 5))
state_totals.plot(kind='bar', ax=ax)
ax.set_title('Total reports by state over the observed period')
ax.set_xlabel('State code / unknown bucket'); ax.set_ylabel('Accident reports')
plt.tight_layout(); plt.show()

## 4. Validate the panel and freeze the common chronological split

The split is made on `week_id`, never by random row sampling. The first 80% of ordered weeks are training; the last 20% are final test. The same boundary is used by both streams.

The 3I and 3N notebooks may read the full historical series to construct *past-only* lags/windows for test forecasting. All fitted transformations (scalers, model weights, target transforms) are fitted using the training portion only. Optuna validation folds are also entirely inside the training portion.

In [ ]:
TRAIN_FRACTION = 0.80
N_WEEKS = int(len(national))
N_TRAIN_WEEKS = int(np.floor(N_WEEKS * TRAIN_FRACTION))
if N_TRAIN_WEEKS < 120 or N_WEEKS - N_TRAIN_WEEKS < 20:
    raise ValueError('Insufficient weekly history for the planned 52-week features and test evaluation.')

train_end_week_id = int(national.iloc[N_TRAIN_WEEKS - 1]['week_id'])
first_test_week_id = train_end_week_id + 1
train_national = national[national.week_id <= train_end_week_id].copy()
test_national = national[national.week_id >= first_test_week_id].copy()

split_metadata = {
    'train_fraction': TRAIN_FRACTION,
    'n_weeks': N_WEEKS,
    'n_train_weeks': int(len(train_national)),
    'n_test_weeks': int(len(test_national)),
    'train_end_week_id': train_end_week_id,
    'first_test_week_id': first_test_week_id,
    'first_week_start': str(national.week_start.min().date()),
    'last_week_start': str(national.week_start.max().date()),
    'frequency': 'weekly',
    'week_definition': 'Monday-start local-calendar week based on source Start_Time',
    'forecast_horizon_intervals': 1,
    'evaluation_protocol': 'frozen fitted weights; rolling one-week-ahead forecasts using observed history through each forecast origin',
    'target': 'national_accident_count = sum of all state counts, including __UNKNOWN__ if present',
    'random_split': False,
}

panel.to_parquet(ARTIFACT_DIR / 'weekly_state_panel.parquet', index=False)
national.to_parquet(ARTIFACT_DIR / 'weekly_national_counts.parquet', index=False)
week_index.to_parquet(ARTIFACT_DIR / 'weekly_index.parquet', index=False)
train_national.to_parquet(ARTIFACT_DIR / 'weekly_national_train.parquet', index=False)
test_national.to_parquet(ARTIFACT_DIR / 'weekly_national_test.parquet', index=False)
(ARTIFACT_DIR / 'split_metadata.json').write_text(json.dumps(split_metadata, indent=2))
(ARTIFACT_DIR / 'state_nodes.json').write_text(json.dumps({'states': states}, indent=2))

print(json.dumps(split_metadata, indent=2))
print('Saved weekly panel, national series, split metadata and state node list.')

## 5. Final checks

The following checks must pass before continuing:

- National weekly totals sum to the number of valid timestamped reports from Notebook 1.
- `week_id` is consecutive, unique at the national level, and sorted chronologically.
- The training weeks precede every test week.
- All state rows—including `__UNKNOWN__` where present—are included in the panel.

In [ ]:
assert national.week_id.is_monotonic_increasing
assert national.week_id.is_unique
assert np.array_equal(national.week_id.to_numpy(), np.arange(1, N_WEEKS + 1))
assert train_national.week_id.max() < test_national.week_id.min()
assert panel.accident_count.sum() == national.national_accident_count.sum()
assert panel.groupby('week_id').state.nunique().eq(len(states)).all()
print('All Notebook 2 checks passed.')